# Whisper fine-tuning (QLoRA) — dialect + task-specific dataset

## 1. Install dependencies

In [ ]:
!pip install --upgrade pip

!pip install transformers
!pip install peft==0.10.0
!pip install trl==0.8.6
!pip install accelerate
!pip install datasets[audio]
!pip install evaluate
!pip install jiwer
!pip install tensorboard

!python -m pip install bitsandbytes --prefer-binary --extra-index-url=https://jllllll.github.io/bitsandbytes-windows-webui

## 2. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 3. Paths & config

In [ ]:
import os, sys

DRIVE_ROOT       = '/content/drive/MyDrive'
BASE_DIR         = f'{DRIVE_ROOT}/capstone_design'
SRC_DIR          = f'{BASE_DIR}/whisper_src'
BASE_MODEL       = 'openai/whisper-large-v3-turbo'
WHISPER_DATA_DIR = f'{DRIVE_ROOT}/whisper'
TOTAL_JSON       = f'{BASE_DIR}/data/total_whisper.json'
AUDIO_DIR        = f'{BASE_DIR}/data/audio_files'
OUTPUT_DIR       = '/content/whisper-dialect-turbo-qlora'
DRIVE_OUTPUT_DIR = f'{BASE_DIR}/outputs'
DEVICE_MAP       = None

assert os.path.isfile(f'{SRC_DIR}/whisper_sft.py'), f'Missing: {SRC_DIR}/whisper_sft.py'
assert os.path.isfile(TOTAL_JSON), f'Missing: {TOTAL_JSON}'
assert os.path.isdir(AUDIO_DIR), f'Missing: {AUDIO_DIR}'

if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)

os.makedirs(OUTPUT_DIR, exist_ok=True)
print('Paths OK. Output:', OUTPUT_DIR)

## 4. Load Whisper + LoRA

In [ ]:
import torch
from peft import LoraConfig, prepare_model_for_kbit_training
from transformers import (BitsAndBytesConfig, WhisperForConditionalGeneration,
                          WhisperProcessor, WhisperTokenizer)
from whisper_sft import WhisperTuner

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Using device:', device)

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type='nf4',
)

model = WhisperForConditionalGeneration.from_pretrained(
    BASE_MODEL,
    quantization_config=quantization_config,
    device_map=DEVICE_MAP,
)
model.generation_config.language = 'korean'
model.generation_config.task = 'transcribe'

tokenizer = WhisperTokenizer.from_pretrained(BASE_MODEL, language='korean', task='transcribe')
processor = WhisperProcessor.from_pretrained(BASE_MODEL, language='korean', task='transcribe')

peft_config = LoraConfig(
    lora_alpha=16,
    lora_dropout=0,
    r=16,
    bias='none',
    task_type='SEQ_2_SEQ_LM',
    target_modules=['q_proj', 'v_proj', 'k_proj', 'out_proj'],
)

model = prepare_model_for_kbit_training(model)
model = WhisperTuner(model, peft_config)
print('Model + LoRA ready.')

## 5. Dataset

In [ ]:
from whisper_sft import (load_json_dataset, split_dataset_with_task_focus,
                         preprocess_datasets, validate_dataset)

dataset = load_json_dataset(TOTAL_JSON)

train_dataset, test_dataset = split_dataset_with_task_focus(dataset)

print(f'Train: {len(train_dataset)} | Test: {len(test_dataset)}')

train_dataset, test_dataset = preprocess_datasets(train_dataset, test_dataset, processor)

validate_dataset(train_dataset, 'Train dataset')
validate_dataset(test_dataset, 'Test dataset')

## 6. TensorBoard

In [ ]:
%load_ext tensorboard
%tensorboard --logdir '{OUTPUT_DIR}/runs'

## 7. Train

In [ ]:
from whisper_sft import build_trainer

trainer = build_trainer(
    model, processor, tokenizer, train_dataset, test_dataset, OUTPUT_DIR,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=2,
    learning_rate=2e-5,
    num_train_epochs=20,
    eval_strategy='epoch',
    save_strategy='epoch',
    logging_steps=10,
    lr_scheduler_type='constant_with_warmup',
)

trainer.train()

## 8. Save & copy to Drive

In [ ]:
import os

processor.save_pretrained(OUTPUT_DIR)
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

os.makedirs(DRIVE_OUTPUT_DIR, exist_ok=True)
!ls -lh {OUTPUT_DIR}
!cp -r {OUTPUT_DIR} {DRIVE_OUTPUT_DIR}
print('Copied ->', DRIVE_OUTPUT_DIR)